In [2]:
import polars as pl
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, when

In [3]:
df = pl.read_parquet("yellow_tripdata_2026-03.parquet")
df.head()


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2026-03-01 00:02:26,2026-03-01 00:13:45,1,2.58,1,"""N""",48,151,1,14.2,1.0,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
2,2026-03-01 00:19:33,2026-03-01 00:28:21,1,1.5,1,"""N""",238,166,1,10.0,1.0,0.5,0.0,0.0,1.0,15.0,2.5,0.0,0.0
2,2026-03-01 00:07:20,2026-03-01 00:15:12,2,0.88,1,"""N""",90,249,1,8.6,1.0,0.5,2.87,0.0,1.0,17.22,2.5,0.0,0.75
2,2026-03-01 00:16:11,2026-03-01 00:28:20,1,1.76,1,"""N""",249,137,1,12.1,1.0,0.5,3.57,0.0,1.0,21.42,2.5,0.0,0.75
2,2026-03-01 00:20:47,2026-03-01 00:30:44,2,1.57,1,"""N""",100,142,1,11.4,1.0,0.5,3.43,0.0,1.0,20.58,2.5,0.0,0.75


In [4]:
print(df.shape)
print(df.columns)

(3952451, 20)
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


***Pre-Processing***

Feature selection

In [5]:
df = df.select([
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "payment_type"
])
df

tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,payment_type
datetime[μs],datetime[μs],i32,i32,i64,f64,f64,f64,f64,i64
2026-03-01 00:02:26,2026-03-01 00:13:45,48,151,1,2.58,14.2,4.99,24.94,1
2026-03-01 00:19:33,2026-03-01 00:28:21,238,166,1,1.5,10.0,0.0,15.0,1
2026-03-01 00:07:20,2026-03-01 00:15:12,90,249,2,0.88,8.6,2.87,17.22,1
2026-03-01 00:16:11,2026-03-01 00:28:20,249,137,1,1.76,12.1,3.57,21.42,1
2026-03-01 00:20:47,2026-03-01 00:30:44,100,142,2,1.57,11.4,3.43,20.58,1
…,…,…,…,…,…,…,…,…,…
2026-03-31 23:10:43,2026-03-31 23:33:02,13,263,null,8.47,50.14,0.0,60.37,0
2026-03-31 23:41:22,2026-03-31 23:52:36,24,42,null,1.99,12.32,0.0,13.82,0
2026-03-31 23:39:38,2026-04-01 00:13:44,62,237,null,10.21,47.95,0.0,52.7,0


Cleaning

In [ ]:
df = (
    df
    .drop_nulls([
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "passenger_count",
        "tip_amount"
    ]).filter(pl.col("trip_distance") > 0)
    .filter(pl.col("fare_amount") > 0)
    .filter(pl.col("total_amount") > 0)
    .filter(pl.col("passenger_count").is_between(1, 6))
    .filter(pl.col("PULocationID") > 0)
    .filter(pl.col("DOLocationID") > 0)
    .filter(pl.col("trip_distance") < 167)
    .filter(pl.col("tip_amount") >= 0)
)

In [7]:
df

tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,payment_type
datetime[μs],datetime[μs],i32,i32,i64,f64,f64,f64,f64,i64
2026-03-01 00:02:26,2026-03-01 00:13:45,48,151,1,2.58,14.2,4.99,24.94,1
2026-03-01 00:19:33,2026-03-01 00:28:21,238,166,1,1.5,10.0,0.0,15.0,1
2026-03-01 00:07:20,2026-03-01 00:15:12,90,249,2,0.88,8.6,2.87,17.22,1
2026-03-01 00:16:11,2026-03-01 00:28:20,249,137,1,1.76,12.1,3.57,21.42,1
2026-03-01 00:20:47,2026-03-01 00:30:44,100,142,2,1.57,11.4,3.43,20.58,1
…,…,…,…,…,…,…,…,…,…
2026-03-31 23:17:20,2026-03-31 23:27:29,236,163,1,2.29,12.8,4.64,23.19,1
2026-03-31 23:30:28,2026-03-31 23:40:04,163,246,1,1.9,11.4,3.43,20.58,1
2026-03-31 23:15:21,2026-03-31 23:30:37,142,107,1,2.62,16.3,2.22,24.27,1


In [8]:
df.write_parquet("cleaned_taxi_data.parquet")

***Automating Pipeline (PySpark)***

In [9]:
spark = (
    SparkSession.builder
    .appName("Taxi Fare Prediction")
    .master("local[2]")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

In [10]:
df_spark = spark.read.parquet("cleaned_taxi_data.parquet")

df_spark.show(5)

+--------------------+---------------------+------------+------------+---------------+-------------+-----------+----------+------------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|tip_amount|total_amount|payment_type|
+--------------------+---------------------+------------+------------+---------------+-------------+-----------+----------+------------+------------+
| 2026-03-01 00:02:26|  2026-03-01 00:13:45|          48|         151|              1|         2.58|       14.2|      4.99|       24.94|           1|
| 2026-03-01 00:19:33|  2026-03-01 00:28:21|         238|         166|              1|          1.5|       10.0|       0.0|        15.0|           1|
| 2026-03-01 00:07:20|  2026-03-01 00:15:12|          90|         249|              2|         0.88|        8.6|      2.87|       17.22|           1|
| 2026-03-01 00:16:11|  2026-03-01 00:28:20|         249|         137|              1|         1.76|

**Transformation**

In [11]:
df_spark = (
    df_spark
    .withColumn("pickup_hour" ###new
                , F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_day" ###new
                , F.dayofweek("tpep_pickup_datetime"))
    .withColumn("pickup_date"
                , F.to_date("tpep_pickup_datetime"))
    .withColumn(
        "trip_duration_minutes", ###new
        (
            F.unix_timestamp("tpep_dropoff_datetime") -
            F.unix_timestamp("tpep_pickup_datetime")
        ) / 60
    )
    .withColumn(
        "revenue_per_mile", ###new
        F.col("total_amount") / F.col("trip_distance")
    )
)

df_spark = (df_spark
    .filter(F.col("trip_duration_minutes") > 0)
    .filter(F.col("trip_duration_minutes") < 300)
    .filter(F.col("revenue_per_mile") > 0)
    .filter(F.col("pickup_hour").between(0, 23)))


df_spark.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pickup_hour",
    "pickup_day",
    "pickup_date",
    "trip_duration_minutes",
    "revenue_per_mile"
).show(10)

+--------------------+---------------------+-----------+----------+-----------+---------------------+------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|pickup_hour|pickup_day|pickup_date|trip_duration_minutes|  revenue_per_mile|
+--------------------+---------------------+-----------+----------+-----------+---------------------+------------------+
| 2026-03-01 00:02:26|  2026-03-01 00:13:45|          0|         1| 2026-03-01|   11.316666666666666| 9.666666666666666|
| 2026-03-01 00:19:33|  2026-03-01 00:28:21|          0|         1| 2026-03-01|                  8.8|              10.0|
| 2026-03-01 00:07:20|  2026-03-01 00:15:12|          0|         1| 2026-03-01|    7.866666666666666|19.568181818181817|
| 2026-03-01 00:16:11|  2026-03-01 00:28:20|          0|         1| 2026-03-01|                12.15|12.170454545454547|
| 2026-03-01 00:20:47|  2026-03-01 00:30:44|          0|         1| 2026-03-01|                 9.95|13.108280254777068|
| 2026-03-01 00:53:06|  2026-03-

In [12]:
df_spark = df_spark.withColumn(
    "payment_type_label",
    when(col("payment_type") == 0, "Flex fare trip")
    .when(col("payment_type") == 1, "Credit card")
    .when(col("payment_type") == 2, "Cash")
    .when(col("payment_type") == 3, "No charge")
    .when(col("payment_type") == 4, "Dispute")
    .when(col("payment_type") == 5, "Unknown")
    .when(col("payment_type") == 6, "Voided trip")
    .otherwise("Other")
)

**Aggregation**

Demand by hour

In [13]:
demand_by_hour_spark = (
    df_spark
    .groupBy("pickup_hour")
    .agg(
        F.count("*").alias("number_of_trips"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance")
    )
    .orderBy(F.col("number_of_trips").desc())
)

demand_by_hour_spark.show(24)



+-----------+---------------+----------------+-----------------+
|pickup_hour|number_of_trips|avg_total_amount|avg_trip_distance|
+-----------+---------------+----------------+-----------------+
|         18|         206853|           28.44|             2.96|
|         17|         204466|           29.92|             3.16|
|         16|         188829|           31.61|              3.5|
|         15|         184794|           30.21|             3.58|
|         19|         182272|           28.82|             3.18|
|         14|         175593|           29.96|             3.55|
|         21|         172164|           28.96|             3.51|
|         20|         169519|           28.57|             3.44|
|         13|         162330|           28.63|             3.34|
|         12|         157453|           28.07|             3.24|
|         11|         146647|           27.68|             3.15|
|         22|         142476|           30.28|             3.78|
|         10|         136

Demand by zone

In [14]:
demand_by_zone_spark = (
    df_spark
    .groupBy("PULocationID")
    .agg(
        F.count("*").alias("number_of_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance")
    )
    .orderBy(F.col("number_of_trips").desc())
)

demand_by_zone_spark.show(20)

+------------+---------------+-------------+------------+
|PULocationID|number_of_trips|total_revenue|avg_distance|
+------------+---------------+-------------+------------+
|         237|         152325|   3179055.95|        1.74|
|         132|         146233|1.177592786E7|       15.16|
|         161|         137475|   3554079.86|        2.28|
|         236|         131394|   2768660.82|        1.89|
|         186|         106398|   2729828.56|        2.25|
|         162|         104870|   2612218.64|        2.21|
|         230|          93215|   2648260.85|        2.85|
|         142|          92565|   2071725.49|         2.1|
|         138|          91954|   6443159.16|        9.61|
|         163|          83605|   2101680.55|        2.28|
|         170|          80490|   1989457.24|        2.28|
|         234|          79062|   1853843.97|        2.02|
|         239|          78269|   1698245.39|        2.07|
|          68|          75347|   1966786.19|        2.47|
|         141|

Payment type analysis

In [22]:
payment_analysis_spark = (
    df_spark
    .groupBy("payment_type_label")
    .agg(
        F.count("*").alias("number_of_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip")
    )
    .orderBy(F.col("total_revenue").desc())
)

payment_analysis_spark.show()

+------------------+---------------+-------------+-------+
|payment_type_label|number_of_trips|total_revenue|avg_tip|
+------------------+---------------+-------------+-------+
|       Credit card|        2535272|7.540784707E7|   4.14|
|              Cash|         332039|   8320241.44|    0.0|
|           Dispute|          15548|    430615.29|   0.01|
|         No charge|           7315|    172307.61|    0.0|
+------------------+---------------+-------------+-------+



***Pipeline***

Splitting

In [16]:
train_df, test_df = df_spark.randomSplit([0.8, 0.2], seed=42)

In [17]:
# Stage 1 — Assemble selected numeric columns into one vector
assembler = VectorAssembler(
    inputCols=[
        "passenger_count",
        "trip_distance",
        "trip_duration_minutes",
        "revenue_per_mile",
        "pickup_hour"
    ],
    outputCol="features",
    handleInvalid="skip"
)

# Stage 2 — Scale the feature vector
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

# Build the preprocessing pipeline
prep_pipeline = Pipeline(stages=[
    assembler,
    scaler
])

# Fit on training data only
prep_model = prep_pipeline.fit(train_df)

# Transform both train and test using the same learned scaling values
clean_train = prep_model.transform(train_df)
clean_test = prep_model.transform(test_df)

# Inspect the result
clean_train.select(
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "revenue_per_mile",
    "pickup_hour",
    "features",
    "scaled_features"
).show(5, truncate=False)

+---------------+-------------+---------------------+------------------+-----------+-----------------------------------------------------+--------------------------------------------------------------------------------------------------------+
|passenger_count|trip_distance|trip_duration_minutes|revenue_per_mile  |pickup_hour|features                                             |scaled_features                                                                                         |
+---------------+-------------+---------------------+------------------+-----------+-----------------------------------------------------+--------------------------------------------------------------------------------------------------------+
|2              |5.13         |29.116666666666667   |8.432748538011696 |23         |[2.0,5.13,29.116666666666667,8.432748538011696,23.0] |[1.2205083837577262,0.36292034346451196,0.7535364591424893,-0.06230754791569481,1.5315623027157828]     |
|1              |1.96   

**Linear Regression advanced predictive analysis**

In [18]:
# Create the Linear Regression estimator
lr = LinearRegression(featuresCol='scaled_features', labelCol='fare_amount', maxIter=10)

# Train the model
lr_model = lr.fit(clean_train)

# View the learned coefficients
print("Coefficients:", lr_model.coefficients)
print("Intercept:", lr_model.intercept)

Coefficients: [0.4098275219690575,14.740775256606081,2.4022565198964476,2.3575603901708866,0.07432319535417592]
Intercept: 19.594486174122462


In [19]:
# Make predictions
predictions = lr_model.transform(clean_test)

# View some predictions vs actual values
predictions.select('fare_amount', 'prediction', 'trip_distance', 'trip_duration_minutes').show(10)

+-----------+------------------+-------------+---------------------+
|fare_amount|        prediction|trip_distance|trip_duration_minutes|
+-----------+------------------+-------------+---------------------+
|       11.4|10.913573263549228|         1.08|                11.25|
|       35.0|12.675497259495968|         1.11|   15.983333333333333|
|       26.1| 29.13096837306233|         5.41|                18.75|
|        8.6| 10.80608011498931|         1.28|                 6.85|
|        7.9|  8.57906935632858|         0.62|                  7.3|
|       19.1|17.901928064330235|         3.02|   18.783333333333335|
|       13.5|14.241642894081533|         2.18|                 12.5|
|       28.2|25.943820165697247|         5.22|   25.183333333333334|
|       21.2|18.332214725489933|         3.09|   20.133333333333333|
|       10.7|11.103312424946798|         1.32|                 9.75|
+-----------+------------------+-------------+---------------------+
only showing top 10 rows


In [20]:
# Evaluate the model
ev_mse = RegressionEvaluator(
    labelCol='fare_amount',
    predictionCol='prediction',
    metricName='mse'
)
ev_rmse = RegressionEvaluator(
    labelCol='fare_amount',
    predictionCol='prediction',
    metricName='rmse'
)
ev_mae = RegressionEvaluator(
    labelCol='fare_amount',
    predictionCol='prediction',
    metricName='mae'
)
ev_r2 = RegressionEvaluator(
    labelCol='fare_amount',
    predictionCol='prediction',
    metricName='r2'
)
mse = ev_mse.evaluate(predictions)
rmse = ev_rmse.evaluate(predictions)
mae = ev_mae.evaluate(predictions)
r2 = ev_r2.evaluate(predictions)



print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")

MSE: 45.9255
RMSE: 6.7768
MAE:  2.8342
R²:   0.8599


In [21]:
df_pandas = df_spark.toPandas()

df_pandas.to_parquet("final_taxi_data.parquet",index=False)